# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook đã điền code cho Colab từ bộ khung bài lab. Các ô thí nghiệm chỉ tạo số liệu khi chạy thật;
ô cài đặt, tải dữ liệu và chạy `eval.py` giữ theo khung. Đọc `README.md`, `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Chạy sau khi đưa bản code/ đã hoàn thiện lên Colab; không import starter/.
!pip -q install timm fvcore openpyxl scikit-learn matplotlib
import os, sys, platform
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

REPO_DIR = Path('K4-Track4-Day2-Deeplearning-Advance').resolve()
assert (REPO_DIR / 'eval.py').is_file(), 'Cần clone/copy repo vào Colab trước'
SUBMISSION_DIR = REPO_DIR / 'submissions/2A202602759_NguyenMinhHien'
assert (SUBMISSION_DIR / 'code/train.py').is_file(), 'Cần đưa code bài làm lên Colab trước'
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(SUBMISSION_DIR / 'code'))
from train import Config, run, set_seed
import dataset, model, losses, inference, benchmark
from eval import compute_metrics, save_predictions
print('python', platform.python_version(), '| torch', torch.__version__, '| timm', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU')
assert torch.cuda.is_available(), 'Bài thí nghiệm đầy đủ cần bật GPU trong Colab'

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
LABELS_DIR = str(Path('data/labels').resolve())
sample_name = pd.read_csv(Path(LABELS_DIR) / 'train_subset0.csv').iloc[0]['Filename']
matches = list(Path('data').rglob(str(sample_name)))
assert len(matches) == 1, f'Không tìm thấy chính xác một ảnh mẫu: {sample_name}; matches={matches[:3]}'
IMAGES_DIR = str(matches[0].parent.resolve())
print('Images:', IMAGES_DIR, '| Labels:', LABELS_DIR)

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_stats = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)
counts = pd.concat([frame['Label'].value_counts().rename(name)
                    for name, frame in [('train', train_df), ('val', val_df), ('test', test_df)]], axis=1).fillna(0).astype(int).sort_index()
display(counts)
counts.plot.bar(figsize=(11, 4), title='DeepWeeds fold 0: số ảnh theo lớp')
plt.xlabel('Label'); plt.ylabel('Số ảnh'); plt.tight_layout(); plt.show()
fig, axes = plt.subplots(9, 3, figsize=(8, 20))
for label in range(9):
    for ax, filename in zip(axes[label], train_df.loc[train_df.Label == label, 'Filename'].head(3)):
        ax.imshow(Image.open(Path(IMAGES_DIR) / filename).convert('RGB'))
        ax.set_title(f'{label}: {dataset.CLASS_NAMES[label]}')
        ax.axis('off')
plt.tight_layout(); plt.show()
print('Tỉ lệ lớp lớn nhất / nhỏ nhất:', counts.sum(axis=1).max() / counts.sum(axis=1).min())
print('Đối chiếu Table 1: Negative 9106; các loài cỏ 1009–1125 ảnh (kiểm tra bảng đếm trên).')

In [ ]:
# Kiểm tra loss ban đầu, ảnh augmentation và khả năng overfit 8 ảnh trước khi chạy sweep.
set_seed(0)
device = torch.device('cuda')
tiny = train_df.head(8).copy()
transform = dataset.build_transforms(True, 224, 'basic')
tiny_loader = dataset.make_loader(tiny, IMAGES_DIR, transform, 8, True, num_workers=0)
x, y, names = next(iter(tiny_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, label in zip(axes.flat, x, y):
    raw = image.permute(1, 2, 0).numpy() * dataset.IMAGENET_STD + dataset.IMAGENET_MEAN
    ax.imshow(np.clip(raw, 0, 1)); ax.set_title(dataset.CLASS_NAMES[int(label)]); ax.axis('off')
plt.tight_layout(); plt.show()
probe = model.build_model('resnet50', pretrained=False, init='scratch').to(device)
criterion = torch.nn.CrossEntropyLoss()
with torch.no_grad():
    initial = criterion(probe(x.to(device)), y.to(device)).item()
print('Initial CE:', initial, '| ln(9):', np.log(9))
# Dùng một batch CỐ ĐỊNH không augmentation lại, để kiểm tra vòng gradient.
opt = torch.optim.AdamW(probe.parameters(), lr=1e-3)
for step in range(300):
    probe.train(); opt.zero_grad()
    logits = probe(x.to(device)); loss = criterion(logits, y.to(device))
    loss.backward(); opt.step()
    if (step + 1) % 50 == 0:
        acc = (logits.argmax(1) == y.to(device)).float().mean().item()
        print(step + 1, 'loss', round(loss.item(), 4), 'acc', round(acc, 4))
assert acc >= 0.99, 'Chưa overfit được một batch; dừng và kiểm tra pipeline'
del probe
torch.cuda.empty_cache()

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# Mỗi backbone cùng recipe T00 và cùng seed. Chỉ dùng val để chọn.
backbone_names = ['resnet50', 'resnext50_32x4d', 'convnext_tiny',
                  'deit_small_patch16_224', 'efficientnet_b0']
common = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR,
              out_dir=str(SUBMISSION_DIR / 'runs'), pred_dir=str(SUBMISSION_DIR / 'predictions'),
              epochs=10, batch_size=64, num_workers=2)
backbone_results = []
for index, name in enumerate(backbone_names, 1):
    result = run(Config(exp_id=f'B{index:02}', backbone=name, seed=0, **common))
    timing_net = model.build_model(name, pretrained=False).cuda().eval()
    weights = torch.load(Path(common['out_dir']) / f'B{index:02}' / 'seed0/best.pt', map_location='cuda', weights_only=False)
    timing_net.load_state_dict(weights['model'])
    timing = benchmark.latency_report(timing_net, 1, 224, device='cuda', warmup=10, iters=50)
    for metric in ('p50', 'p95', 'p99', 'images_per_s'):
        result[f'latency_batch1_{metric}_ms' if metric != 'images_per_s' else metric] = timing[metric]
    del timing_net
    torch.cuda.empty_cache()
    backbone_results.append(result)
backbone_table = pd.DataFrame(backbone_results).sort_values('macro_f1_val', ascending=False)
display(backbone_table)
best_backbone = str(backbone_table.iloc[0]['backbone'])
print('Backbone có macro-F1 val cao nhất:', best_backbone, '(kết luận tạm từ 1 seed)')

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# T00 và ba ablation đơn yếu tố, cùng backbone, fold, epoch và seed.
from dataclasses import replace
baseline_cfg = Config(exp_id='T00', backbone=best_backbone, seed=0, **common)
training_results = [dict(run(baseline_cfg), axis='baseline', difference_vs_T00='none')]
ablations = [
    ('T01', {'init': 'frozen'}, 'A: đóng băng'),
    ('T02', {'aug': 'color'}, 'B: tăng cường màu'),
    ('T03', {'loss': 'ls', 'label_smoothing': 0.1}, 'C: label smoothing'),
]
for exp_id, changes, axis in ablations:
    result = run(replace(baseline_cfg, exp_id=exp_id, **changes))
    training_results.append(dict(result, axis=axis, difference_vs_T00=str(changes)))
# T04 kết hợp hai thay đổi đơn có macro-F1 val cao nhất.
ranked = sorted(zip(ablations, training_results[1:]),
                key=lambda pair: pair[1]['macro_f1_val'], reverse=True)
combined_changes = {key: value for (spec, _) in ranked[:2] for key, value in spec[1].items()}
combined = run(replace(baseline_cfg, exp_id='T04', **combined_changes))
training_results.append(dict(combined, axis='kết hợp', difference_vs_T00=str(combined_changes)))
training_table = pd.DataFrame(training_results)
training_table['delta_vs_T00'] = training_table.macro_f1_val - training_table.iloc[0].macro_f1_val
display(training_table)


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# So sánh suy luận trên VAL từ recipe tốt nhất; không đọc test.
val_loader = dataset.make_loader(val_df, IMAGES_DIR, dataset.build_transforms(False, 224),
                                 64, False, num_workers=2)
net = model.build_model(best_backbone, pretrained=False).cuda().eval()
chosen_training_id = pd.DataFrame(training_results).sort_values('macro_f1_val', ascending=False).iloc[0]['exp_id']
checkpoint = torch.load(Path(common['out_dir']) / chosen_training_id / 'seed0/best.pt', map_location='cuda', weights_only=False)
net.load_state_dict(checkpoint['model'])
names, yv, z0 = inference.predict_logits(net, val_loader, 'cuda')
_, _, zflip = inference.predict_logits(net, val_loader, 'cuda', inference.view_hflip)
def center_crop_view(x):
    return torch.nn.functional.interpolate(inference.views_multicrop(x, int(x.shape[-1] * 0.9))[-1], size=224,
                                           mode='bilinear', align_corners=False)
_, _, zcrop = inference.predict_logits(net, val_loader, 'cuda', center_crop_view)
val_loader_256 = dataset.make_loader(val_df, IMAGES_DIR, dataset.build_transforms(False, 256),
                                     64, False, num_workers=2)
names256, y256, z256 = inference.predict_logits(net, val_loader_256, 'cuda')
assert names256 == names and np.array_equal(y256, yv)
methods = {
    'I00': inference.apply_temperature(z0, 1),
    'I01': inference.aggregate_views([z0, zflip], 'prob'),
    'I02': inference.aggregate_views([z0, zcrop], 'prob'),
    'I03': inference.aggregate_views([z0, zflip], 'logit'),
    'I04': inference.apply_temperature(z256, 1.0),
}
inference_results = []
for exp_id, probs in methods.items():
    score = compute_metrics(yv, probs.argmax(1), probs)
    inference_results.append(dict(exp_id=exp_id, macro_f1_val=score['macro_f1'],
                                  top1_val=score['top1'], ece_val=score['ece'],
                                  resolution=256 if exp_id == 'I04' else 224))
display(pd.DataFrame(inference_results))
latency_rows = []
for batch in (1, 32):
    latency_rows.append({'exp_id': 'I00', **benchmark.latency_report(net, batch, 224, device='cuda')})
    latency_rows.append({'exp_id': 'I01', **benchmark.tta_latency(net, 2, batch_size=batch, img_size=224, device='cuda')})
    latency_rows.append({'exp_id': 'I02', **benchmark.tta_latency(net, 2, batch_size=batch, img_size=224, device='cuda', views=[inference.view_identity, center_crop_view])})
    latency_rows.append({'exp_id': 'I03', **benchmark.tta_latency(net, 2, batch_size=batch, img_size=224, device='cuda')})
    latency_rows.append({'exp_id': 'I04', **benchmark.latency_report(net, batch, 256, device='cuda')})
display(pd.DataFrame(latency_rows))
print('I04: đánh giá ở 256 px trên val')

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# Khóa lựa chọn từ val trước khi mở test; không chỉnh sau khi xem test.
import json
final_row = pd.DataFrame(training_results).sort_values('macro_f1_val', ascending=False).iloc[0]
final_exp = final_row['exp_id']
final_changes = next((changes for exp_id, changes, _ in ablations if exp_id == final_exp), {})
if final_exp == 'T04':
    final_changes = combined_changes
method_map = {'I00': 'i00', 'I01': 'hflip_prob', 'I02': 'center_crop',
              'I03': 'hflip_logit', 'I04': 'resolution256'}
latency_batch1 = {row['exp_id']: row['p95'] for row in latency_rows if row['batch'] == 1}
best_f1 = max(row['macro_f1_val'] for row in inference_results)
eligible = [row for row in inference_results if best_f1 - row['macro_f1_val'] <= 0.005]
method_row = min(eligible, key=lambda row: (latency_batch1[row['exp_id']], row['ece_val']))
final_method = method_map[method_row['exp_id']]
lock = {'selection_split': 'val', 'metric': 'macro_f1',
        'inference_tolerance': 0.005, 'inference_tiebreak': 'batch1 p95 then ECE',
        'backbone': best_backbone, 'training_exp_id': final_exp,
        'training_delta': final_changes, 'inference_id': method_row['exp_id'],
        'inference_method': final_method, 'epochs': common['epochs'],
        'batch_size': common['batch_size'], 'seeds': [0, 1, 2],
        'baseline': 'T00 + I00', 'test_access_before_lock': False}
lock_path = SUBMISSION_DIR / 'final_lock.json'
lock_path.write_text(json.dumps(lock, indent=2), encoding='utf-8')
print('FINAL_LOCKED_FROM_VAL', lock)
# Bắt đầu test chỉ sau khi file lock đã được ghi thành công.
assert lock_path.is_file() and json.loads(lock_path.read_text(encoding='utf-8')) == lock
final_results = []
baseline_results = []
for seed in lock['seeds']:
    final_results.append(run(replace(baseline_cfg, exp_id='F01', seed=seed,
                                     save_test_predictions=True, inference_method=final_method,
                                     **final_changes)))
    baseline_results.append(run(replace(baseline_cfg, exp_id='T00', seed=seed,
                                        save_test_predictions=True, inference_method='i00')))
display(pd.DataFrame(final_results))
display(pd.DataFrame(baseline_results))


In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "{SUBMISSION_DIR}/predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out {SUBMISSION_DIR}/eval_out
!python {REPO_DIR}/eval.py score --pred "{SUBMISSION_DIR}/predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out {SUBMISSION_DIR}/eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "{SUBMISSION_DIR}/predictions/F01_seed*_test.csv" \
    --baseline "{SUBMISSION_DIR}/predictions/T00_seed*_test.csv" \
    --final-val "{SUBMISSION_DIR}/predictions/F01_seed*_val.csv" \
    --latency-p95-ms {latency_batch1[lock['inference_id']]} --latency-method proper \
    --test-csv {LABELS_DIR}/test_subset0.csv --val-csv {LABELS_DIR}/val_subset0.csv \
    --labels {LABELS_DIR}/labels.csv --out {SUBMISSION_DIR}/eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# Chỉ tạo bảng sau khi các thí nghiệm tương ứng đã chạy thật.
from openpyxl.styles import Font, PatternFill
required = ['backbone_results', 'training_results', 'inference_results', 'final_results', 'latency_rows']
assert all(name in globals() for name in required), 'Còn thiếu kết quả chạy thật; không tạo bảng giả'
book = SUBMISSION_DIR / 'results.xlsx'
backbones_sheet = pd.DataFrame(backbone_results)
training_sheet = pd.DataFrame(training_results)
training_sheet['delta_vs_T00'] = training_sheet['macro_f1_val'] - training_sheet.loc[training_sheet.exp_id == 'T00', 'macro_f1_val'].iloc[0]
inference_sheet = pd.DataFrame(inference_results)
latency_sheet = pd.DataFrame(latency_rows)
inference_sheet = inference_sheet.merge(latency_sheet.loc[latency_sheet['batch'] == 1, ['exp_id','p50','p95','p99','images_per_s']], on='exp_id', how='left')
backbones_sheet['latency_p50_ms'] = backbones_sheet['latency_batch1_p50_ms']
backbone_latency = dict(zip(backbones_sheet.backbone, backbones_sheet.latency_p50_ms))
training_sheet['latency_p50_ms'] = training_sheet.backbone.map(backbone_latency)  # ước tính từ cùng backbone, 1-view
inference_sheet['latency_p50_ms'] = inference_sheet['p50']
summary_sheet = pd.concat([backbones_sheet, training_sheet, inference_sheet], ignore_index=True).sort_values('macro_f1_val', ascending=False).head(10)
from eval import read_pred
final_rows = []
per_class_rows = []
for exp_id in ('T00', 'F01'):
    for seed in (0, 1, 2):
        pred = read_pred(str(SUBMISSION_DIR / 'predictions' / f'{exp_id}_seed{seed}_test.csv'))
        score = compute_metrics(pred.y_true, pred.y_pred, pred.probs)
        val_pred = read_pred(str(SUBMISSION_DIR / 'predictions' / f'{exp_id}_seed{seed}_val.csv'))
        val_score = compute_metrics(val_pred.y_true, val_pred.y_pred, val_pred.probs)
        configuration = f"{best_backbone}; {final_changes}; {final_method}" if exp_id == 'F01' else f"{best_backbone}; T00; i00"
        final_rows.append(dict(exp_id=exp_id, seed=seed, configuration=configuration, macro_f1_val=val_score['macro_f1'],
                               macro_f1_test=score['macro_f1'], top1_test=score['top1'],
                               ece_test=score['ece']))
        for label in range(9):
            per_class_rows.append(dict(exp_id=exp_id, seed=seed, label=dataset.CLASS_NAMES[label],
                                       support=int(score['support'][label]),
                                       precision=float(score['precision'][label]),
                                       recall=float(score['recall'][label]), f1=float(score['f1'][label])))
final_sheet = pd.DataFrame(final_rows)
final_summary = final_sheet.groupby('exp_id')[['macro_f1_val','macro_f1_test','top1_test','ece_test']].agg(['mean','std'])
display(final_summary)  # std mẫu (ddof=1); đưa các dòng này vào báo cáo
summary_rows = []
for exp_id, group in final_sheet.groupby('exp_id'):
    row = {'exp_id': exp_id, 'seed': 'mean ± std'}
    for metric in ('macro_f1_val', 'macro_f1_test', 'top1_test', 'ece_test'):
        row[metric] = f"{group[metric].mean():.4f} ± {group[metric].std(ddof=1):.4f}"
    summary_rows.append(row)
final_sheet = pd.concat([final_sheet, pd.DataFrame(summary_rows)], ignore_index=True)
with pd.ExcelWriter(book, engine='openpyxl') as writer:
    for sheet, frame in [('Backbones', backbones_sheet), ('Training', training_sheet),
                         ('Inference', inference_sheet), ('Final', final_sheet),
                         ('PerClass', pd.DataFrame(per_class_rows)), ('Latency', latency_sheet),
                         ('Summary', summary_sheet)]:
        frame.to_excel(writer, sheet_name=sheet, index=False)
        ws = writer.sheets[sheet]
        ws.freeze_panes = 'A2'
        for cell in ws[1]:
            cell.font = Font(bold=True, color='FFFFFF')
            cell.fill = PatternFill('solid', fgColor='17365D')
print('Đã tạo', book)